<a href="https://colab.research.google.com/github/krutarth3238/slm-lora-finetuning/blob/main/GPT_2_LoRA_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **LoRA Fine Tuning Using GPT-2**

Checking Perplexity for the 10 Dataset Model (Significantly Lower)

In [ ]:
import math

eval_results = trainer.evaluate()
perplexity = math.exp(eval_results["eval_loss"])
print("Perplexity:", perplexity)

Checking the 10 Dataset Model with a Sample Prompt (Better Result then 1 Dataset Model)

In [ ]:
prompt = "My Name is Barry Allen and I moonlight as"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

start = time.time()
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=True,
    temperature=0.8,
    top_p=0.9
)
end = time.time()

generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

print("Generated Text:\n")
print(generated_text)

latency = end - start
tokens_per_sec = 50 / latency

print("\nLatency:", latency)
print("Tokens/sec:", tokens_per_sec)


Checking Latency for 10 Dataset Model

In [ ]:
import time



latency = end - start
tokens_per_sec = 100 / latency

print("Latency:", latency)
print("Tokens/sec:", tokens_per_sec)

wandb.log({
    "inference/latency": latency,
    "inference/tokens_per_sec": tokens_per_sec
})


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["c_attn"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)


Importing 10 Datasets Configuring Lora (r=4), Training and Saving the Model

In [ ]:
from datasets import load_dataset, concatenate_datasets
from transformers import TrainingArguments, Trainer
import wandb

from peft import LoraConfig, get_peft_model

from datasets import load_dataset, concatenate_datasets

from datasets import load_dataset, concatenate_datasets

from datasets import load_dataset, concatenate_datasets

def load_and_prepare_dataset(
    name,
    subset=None,
    text_column="text",
    split="train",
    sample_size=800
):
    ds = load_dataset(
        name,
        subset,
        split=split,
        keep_in_memory=True
    )

    ds = ds.shuffle(seed=42)
    ds = ds.select(range(min(sample_size, len(ds))))

    if text_column != "text":
        ds = ds.rename_column(text_column, "text")

    ds = ds.remove_columns([c for c in ds.column_names if c != "text"])
    return ds


datasets_list = []

datasets_list.append(load_and_prepare_dataset(
    "wikitext", "wikitext-2-raw-v1", "text"
))

datasets_list.append(load_and_prepare_dataset(
    "wikitext", "wikitext-103-raw-v1", "text"
))

datasets_list.append(load_and_prepare_dataset(
    "roneneldan/TinyStories", None, "text"
))

datasets_list.append(load_and_prepare_dataset(
    "ag_news", None, "text"
))

datasets_list.append(load_and_prepare_dataset(
    "xsum", None, "document"
))

datasets_list.append(load_and_prepare_dataset(
    "cnn_dailymail", "3.0.0", "article"
))

datasets_list.append(load_and_prepare_dataset(
    "squad", None, "context"
))

datasets_list.append(load_and_prepare_dataset(
    "yelp_review_full", None, "text"
))

datasets_list.append(load_and_prepare_dataset(
    "imdb", None, "text"
))

datasets_list.append(load_and_prepare_dataset(
    "Menlo/Instruction-text-only-full", None, "text"
))


combined_dataset = concatenate_datasets(datasets_list)
print("Total combined samples:", len(combined_dataset))

def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = combined_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)


split_dataset = tokenized_dataset.train_test_split(
    test_size=0.1,   # 90% train, 10% validation
    seed=42
)

train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["c_attn"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


training_args = TrainingArguments(
    output_dir="./gpt2-lora-multidata",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    report_to="wandb",
    run_name="gpt2_lora_10datasets"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

trainer.save_model("./final_multidata_model")
tokenizer.save_pretrained("./final_multidata_model")


# **The whole Process for one Dataset now we move to Multiple Datasets**



Saving the Model




In [ ]:
trainer.save_model("./final_model")
tokenizer.save_pretrained("./final_model")


Checking Inference Time and Latency

In [ ]:
import time

start = time.time()
model.generate(**inputs, max_new_tokens=100)
end = time.time()

tokens_generated = 100
latency = end - start
tokens_per_sec = tokens_generated / latency

print("Latency:", latency)
print("Tokens/sec:", tokens_per_sec)

wandb.log({
    "inference/latency": latency,
    "inference/tokens_per_sec": tokens_per_sec
})


Testing out the Model with a Sample Prompt

In [ ]:
prompt = "Hello I am the Flash and"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

output = model.generate(**inputs, max_new_tokens=50)

print(tokenizer.decode(output[0], skip_special_tokens=True))


Checking Perplexity of the Model

In [ ]:
import math

eval_results = trainer.evaluate()
perplexity = math.exp(eval_results["eval_loss"])
print("Perplexity:", perplexity)


Training a Dataset (WikiText) using LoRA


In [ ]:
from datasets import load_dataset
from transformers import TrainingArguments
from transformers import Trainer

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
small_train = dataset["train"].shuffle(seed=42).select(range(5000))
small_val = dataset["validation"].select(range(1000))

def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens


tokenized_train = small_train.map(tokenize_function, batched=True, remove_columns=["text"])
tokenized_val = small_val.map(tokenize_function, batched=True, remove_columns=["text"])

training_args = TrainingArguments(
    output_dir="./gpt2-lora",
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    report_to="wandb",
    run_name="gpt2_lora_wikitext"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val
)

trainer.train()



Configuring LoRA Parameters for 1 Dataset

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],  # GPT2 attention layers
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


Importing GPT2 Model from Transformers Library

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


WandB Login to Conduct Experiments

In [ ]:
import wandb
wandb.login()
